In [1]:
from pathlib import Path
import pandas as pd
import metrics
from experiments import Model
from baseline import BaselineModel

path = Path("data/train.csv")
df = pd.read_csv(path)

In [2]:
X = df.drop(columns=["target"])
y = df["target"]

start = {"scale": False, "noize": True, "metric": "euclidean", "agg": "mode", "k": 5,
         "reduce": "top", "n_features": None, "n_objects": None}

variants = [
    ("baseline (медиана)", BaselineModel, {}),
    ("knn k=5, без скейлинга", Model, {**start}),
    ("knn k=5, стандартизация всех 160", Model, {**start, "scale": True}),
    ("knn k=5, без скейлинга + без шума", Model, {**start, "noize": False}),
    ("knn k=5, без шума + стандартизация - eucl", Model, {**start, "scale": True, "noize": False}),
    ("knn k=5, без шума + стандартизация - manh", Model, {**start, "scale": True, "noize": False,
                                                          "metric": "manhattan"}),
    ("knn k=5, без шума + стандартизация - cos", Model, {**start, "scale": True, "noize": False,
                                                         "metric": "cosine"}),
    ("knn k=5, без шума + стандартизация - eucl - median", Model, {**start, "scale": True, "noize": False,
                                                                   "agg": "median"}),
    ("knn k=5, без шума + стандартизация - manh - median", Model, {**start, "scale": True, "noize": False,
                                                                   "metric": "manhattan", "agg": "median"}),
    ("Model() по умолчанию — текущая лучшая", Model, {}),
]

rows = []
for name, model_cls, params in variants:
    mean, std = metrics.mae(model_cls, X, y, **params)
    rows.append({"вариант": name, "mae": round(mean, 4), "std": round(std, 4)})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,вариант,mae,std
0,baseline (медиана),0.8968,0.0002
1,"knn k=5, без скейлинга",0.8411,0.0023
2,"knn k=5, стандартизация всех 160",0.7551,0.0076
3,"knn k=5, без скейлинга + без шума",0.8126,0.0067
4,"knn k=5, без шума + стандартизация - eucl",0.6859,0.0111
5,"knn k=5, без шума + стандартизация - manh",0.6645,0.0075
6,"knn k=5, без шума + стандартизация - cos",0.6983,0.0139
7,"knn k=5, без шума + стандартизация - eucl - median",0.6161,0.0065
8,"knn k=5, без шума + стандартизация - manh - median",0.5948,0.0048
9,Model() по умолчанию — текущая лучшая,0.2391,0.0045


In [3]:
base = {"scale": True, "noize": False, "metric": "manhattan", "agg": "median",
        "reduce": "top", "n_features": None, "n_objects": None}
ks = [5, 11, 21, 31, 51, 75, 101]

rows = []
for k in ks:
    mean, std = metrics.mae(Model, X, y, **base, k=k)
    rows.append({"k": k, "mae": mean, "std": std})

k_results = pd.DataFrame(rows)
display(k_results.round(4))

,k,mae,std
0,5,0.5948,0.0048
1,11,0.5563,0.0058
2,21,0.5451,0.0031
3,31,0.5443,0.0067
4,51,0.5481,0.0036
5,75,0.5545,0.0028
6,101,0.5607,0.0027
